In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Could not find the project root containing the src folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import download_dataset, load_dataset

DATASET_PATH = PROJECT_ROOT / "data" / "heart_disease.csv"

# Download the dataset if it is not already available, then load it.
download_dataset(DATASET_PATH)
df_raw = load_dataset(DATASET_PATH)

print("First five rows before numeric validation:")
display(df_raw.head())

# These columns are expected to contain numeric values or numeric category codes.
# Keep the original values so we can report any entries that fail conversion.
df = df_raw.copy()
invalid_values_found = False

for col in df_raw.columns:
    converted_col = pd.to_numeric(df_raw[col], errors="coerce")
    invalid_mask = df_raw[col].notna() & converted_col.isna()

    if invalid_mask.any():
        invalid_values_found = True
        print(f"\nValues that could not be converted to numbers in '{col}':")
        print(df_raw.loc[invalid_mask, col].value_counts().head(10))

        # Show source row numbers (the CSV has no header row).
        source_rows = (df_raw.index[invalid_mask] + 1).tolist()
        print("CSV row numbers (first 10):", source_rows[:10])

    # Use the numeric version for the remaining EDA.
    df[col] = converted_col

if not invalid_values_found:
    print("\nNo non-empty, non-numeric values were found.")

print("\nData types after conversion:")
print(df.dtypes)

print("\nMissing values after conversion:")
missing_values = df.isna().sum()
print(missing_values[missing_values > 0] if missing_values.any() else "None")

print("\nSummary statistics:")
display(df.describe().T)

print("\nOriginal target counts:")
print(df["target"].value_counts(dropna=False).sort_index())

print("\nOriginal target proportions:")
print(df["target"].value_counts(normalize=True, dropna=False).sort_index())

In [ ]:
df = load_dataset(DATASET_PATH)
df = df.replace("?", pd.NA).apply(pd.to_numeric, errors="coerce")

In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nSummary:")
display(df.describe().T)

In [ ]:
coded_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]
for col in coded_cols:
    print(col, sorted(df[col].dropna().unique()))

In [ ]:
print(df["target"].value_counts(dropna=False).sort_index())

# In this dataset, 0 means no disease and 1-4 indicate disease presence.
df["target"] = (df["target"] > 0).astype(int)

print("\nBinary target counts:")
print(df["target"].value_counts().sort_index())
print("\nBinary target proportions:")
print(df["target"].value_counts(normalize=True).sort_index())

In [ ]:
sns.countplot(data=df, x="target")
plt.title("Heart Disease Class Counts")
plt.xlabel("Target (0 = absent, 1 = present)")
plt.ylabel("Number of records")
plt.show()

In [ ]:
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for col, ax in zip(numeric_cols, axes.flat):
    sns.histplot(data=df, x=col, discrete=(col == "ca"), kde=(col != "ca"), ax=ax)
    ax.set_title(f"Distribution of {col}")

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, ax in zip(numeric_cols, axes.flat):
    sns.boxplot(
        data=df,
        x="target",
        y=col,
        order=[0, 1],
        ax=ax,
    )
    ax.set_title(f"{col} by heart disease class")
    ax.set_xlabel("Target (0 = absent, 1 = present)")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
heatmap_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca", "target"]

plt.figure(figsize=(10, 7))
sns.heatmap(
    df[heatmap_cols].corr(),
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f",
)
plt.title("Correlation Among Numeric Features and Binary Target")
plt.tight_layout()
plt.show()